# Q-Network (Dueling DQN)

PyTorch MLP with dueling streams.

```
Input (state_dim)
    └── Shared encoder: Linear → ReLU → Linear → ReLU  (128 units)
            ├── Value head     → V(s)           scalar
            └── Advantage head → A(s, a)        n_actions
                    Q(s,a) = V(s) + A(s,a) - mean_a[A(s,a)]
```

Dueling DQN separates state value from action advantage, improving stability especially in states where the choice of action matters little.

In [1]:
import torch
import torch.nn as nn

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.9.1+cu128


In [2]:
class QNetwork(nn.Module):
    def __init__(
        self,
        state_dim:  int,
        n_actions:  int,
        hidden_dim: int  = 128,
        dueling:    bool = True,
    ):
        super().__init__()
        self.dueling = dueling

        self.encoder = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        if dueling:
            self.value_head = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 1)
            )
            self.advantage_head = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, n_actions)
            )
        else:
            self.q_head = nn.Sequential(
                nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, n_actions)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x)
        if self.dueling:
            V = self.value_head(h)               # (B, 1)
            A = self.advantage_head(h)           # (B, n_actions)
            return V + (A - A.mean(dim=1, keepdim=True))
        return self.q_head(h)

print("QNetwork defined.")

QNetwork defined.


## Sanity check

In [3]:
state_dim = 11
n_actions = 5

net = QNetwork(state_dim, n_actions)
dummy = torch.randn(4, state_dim)   # batch of 4
q    = net(dummy)

print(f"Input  : {dummy.shape}")
print(f"Output : {q.shape}   (batch × n_actions)")
print(f"Params : {sum(p.numel() for p in net.parameters()):,}")

Input  : torch.Size([4, 11])
Output : torch.Size([4, 5])   (batch × n_actions)
Params : 34,950
